In [1]:
import heapq
import matplotlib.pyplot as plt

# -------------------------------
# 1. Create the grid
# -------------------------------
# 0 = free cell
# 1 = obstacle
grid = [
    [0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 0, 0, 0]
]
start = (0, 0)
goal = (4, 4)

# -------------------------------
# 2. Heuristic function
# -------------------------------
def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

# -------------------------------
# 3. A* algorithm
# -------------------------------
def astar(grid, start, goal):
    rows = len(grid)
    cols = len(grid[0])
    
    # Priority queue
    open_list = []
    
    # Add start node
    heapq.heappush(open_list, (0, start))
    
    # Remember where each node came from
    came_from = {}
    
    # Cost from start
    g_score = {start: 0}
    
    # Estimated total cost
    f_score = {
        start: heuristic(start, goal)
    }
    
    # Four possible movements
    directions = [
        (-1, 0),  # up
        (1, 0),   # down
        (0, -1),  # left
        (0, 1),   # right
    ]

    # ---------------------------------
    # Main loop
    # ---------------------------------
    while open_list:
        # Get node with smallest f-score
        current_f, current = heapq.heappop(open_list)

        # ---------------------------------
        # Goal reached
        # ---------------------------------
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            # Reverse path
            path.reverse()
            return path

        # ---------------------------------
        # Check neighbors
        # ---------------------------------
        r, c = current
        for dr, dc in directions:
            nr = r + dr
            nc = c + dc
            
            # Check neighbor is inside grid and is not an obstacle
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0:
                neighbor = (nr, nc)
                
                # Cost of moving to neighbor
                tentative_g = g_score[current] + 1
                
                # Check if this is a better path
                if tentative_g < g_score.get(neighbor, float("inf")):
                    # Remember previous node
                    came_from[neighbor] = current
                    # Store g-score
                    g_score[neighbor] = tentative_g
                    # Calculate f = g + h
                    f = tentative_g + heuristic(neighbor, goal)     
                    f_score[neighbor] = f

                    # Add neighbor to priority queue
                    heapq.heappush(open_list, (f, neighbor))

    # No path found
    return None

# ------------------------------
# Run A*
# ------------------------------
path = astar(grid, start, goal)
print("Path:")
print(path)

Path:
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4)]


In [2]:
import heapq

# Goal state definition
goal = ((1, 2, 3), 
        (4, 5, 6), 
        (7, 8, 0))

# Pre-compute target positions for Manhattan Distance heuristic
GOAL_POSITIONS = {}
for r in range(3):
    for c in range(3):
        val = goal[r][c]
        if val != 0:
            GOAL_POSITIONS[val] = (r, c)


# Function to locate the empty space (0)
def find_blank(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j


# Convert list to immutable tuple for hashing/visited set
def state_to_tuple(state):
    return tuple(tuple(row) for row in state)


# Convert tuple back to mutable list of lists
def tuple_to_state(state_tuple):
    return [list(row) for row in state_tuple]


# Generate valid next board states
def get_neighbors(state):
    neighbors = []
    x, y = find_blank(state)
    
    # Moves: Up, Down, Left, Right
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        
        # Ensure target tile stays within 3x3 boundaries
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = tuple_to_state(state)
            new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
            neighbors.append(state_to_tuple(new_state))
            
    return neighbors


# Heuristic function: Manhattan Distance
def heuristic(state):
    distance = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                target_r, target_c = GOAL_POSITIONS[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance


# Informed A* Search Algorithm
def astar_solve(start_state, target_state):
    start_tuple = state_to_tuple(start_state) if isinstance(start_state, list) else start_state
    target_tuple = state_to_tuple(target_state) if isinstance(target_state, list) else target_state
    
    if start_tuple == target_tuple:
        return [start_tuple]
    
    # Counter ensures stable priority queue sorting when f_scores are equal
    counter = 0
    
    # Priority Queue storing tuples: (f_score, counter, current_state, path_taken)
    initial_h = heuristic(start_tuple)
    pq = [(initial_h, counter, start_tuple, [start_tuple])]
    
    # Track minimum g_score (actual path cost) for each state
    g_scores = {start_tuple: 0}
    
    while pq:
        f_score, _, current_state, path = heapq.heappop(pq)
        
        if current_state == target_tuple:
            return path
        
        g_cost = len(path) - 1
        
        # If we found a cheaper way to reach this state previously, skip
        if g_cost > g_scores.get(current_state, float('inf')):
            continue
            
        for neighbor in get_neighbors(current_state):
            tentative_g = g_cost + 1
            
            if tentative_g < g_scores.get(neighbor, float('inf')):
                g_scores[neighbor] = tentative_g
                h_cost = heuristic(neighbor)
                f_cost = tentative_g + h_cost
                counter += 1
                heapq.heappush(pq, (f_cost, counter, neighbor, path + [neighbor]))
                
    return None


# Display helper
def print_board(state):
    for row in state:
        print(" ".join(str(val) if val != 0 else "_" for val in row))
    print()


# Driver Execution
if __name__ == "__main__":
    initial_state = [
        [1, 2, 3],
        [4, 0, 6],
        [7, 5, 8]
    ]
    
    print("Initial Board:")
    print_board(initial_state)
    
    path = astar_solve(initial_state, goal)
    
    if path:
        print(f"Solved in {len(path) - 1} steps!\n")
        for step_num, state in enumerate(path):
            print(f"Step {step_num}:")
            print_board(state)
    else:
        print("No solution exists for this configuration.")

Initial Board:
1 2 3
4 _ 6
7 5 8

Solved in 2 steps!

Step 0:
1 2 3
4 _ 6
7 5 8

Step 1:
1 2 3
4 5 6
7 _ 8

Step 2:
1 2 3
4 5 6
7 8 _

